# YOLOv8s traffic sign baseline on Kaggle

Runs the repository pipeline with online 640×640 aspect-preserving resizing, street-grouped train/validation split, augmentation disabled, and configurable inference thresholds. Select a Kaggle GPU accelerator before running.

In [ ]:
from pathlib import Path
import os

DATA_ROOT = Path('/kaggle/input/datasets/ducmonday/zalo-traffic-sign/za_traffic_2020')
REPO_DIR = Path('/kaggle/working/AIT3001.Group6.traffic.sign.detection')
assert (DATA_ROOT / 'traffic_train' / 'annotation.csv').is_file(), f'Cannot find annotation.csv under {DATA_ROOT}'
assert (DATA_ROOT / 'traffic_train' / 'images').is_dir(), f'Cannot find images under {DATA_ROOT}'
print('Dataset:', DATA_ROOT)

In [ ]:
# Kaggle Internet must be enabled for cloning and installing missing packages.
%cd /kaggle/working
!git clone https://github.com/dohoangminh-vn/AIT3001.Group6.traffic.sign.detection.git
%cd /kaggle/working/AIT3001.Group6.traffic.sign.detection
!python -m pip install -r requirements.txt

## Baseline settings

The shared validation inference settings live in `configs/config.yaml`. Change `score_threshold` (`conf`) and `nms_iou` there or pass Hydra overrides in the training command. They are recorded in the run config. This notebook does not sweep thresholds. The baseline keeps augmentation disabled.

In [ ]:
# Optional: set WANDB_API_KEY in Kaggle Secrets, then enable online tracking.
# from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = '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'
# Default to offline so the notebook works without a W&B secret.
TRACKING_MODE = 'online'  # set to 'online' after configuring the secret


In [ ]:
# Prepare grouped split/labels only; image resizing happens online inside each model pipeline.
!python train.py prepare_only=true data.root="{DATA_ROOT}" data.prepared_dir=/kaggle/working/prepared_traffic_signs

In [ ]:
# YOLOv8s baseline, no augmentation. Override epochs/batch/device as needed for the Kaggle GPU.
!python train.py model=yolov8 data.root="{DATA_ROOT}" data.prepared_dir=/kaggle/working/prepared_traffic_signs augmentation.enabled=false model.weights=yolov8s.pt model.imgsz=640 model.epochs=50 model.batch_size=16 model.device=0 evaluation.inference.score_threshold=0.001 evaluation.inference.nms_iou=0.7 tracking.mode={TRACKING_MODE}

## Optional Faster R-CNN run

Run this separately if the GPU session has enough time and memory. It uses the same prepared street split and 640×640 online letterbox geometry; model normalization remains in torchvision.

In [ ]:
# !python train.py model=fasterrcnn data.root="{DATA_ROOT}" data.prepared_dir=/kaggle/working/prepared_traffic_signs augmentation.enabled=false model.imgsz=640 model.epochs=50 model.batch_size=2 model.device=0 tracking.mode={TRACKING_MODE}

In [ ]:
# Package local outputs so they can be downloaded from the Kaggle notebook output panel.
!cd /kaggle/working && zip -qr traffic_sign_outputs.zip AIT3001.Group6.traffic.sign.detection/outputs